# Sintonización fina de modelos de lenguaje con LoRA

Los modelos de lenguaje modernos (LLMs, *Large Language Models*) contienen miles de millones de parámetros y han sido previamente entrenados sobre enormes cantidades de texto provenientes de Internet, libros, artículos y otras fuentes de información.

El entrenamiento completo de estos modelos requiere enormes recursos computacionales, incluyendo múltiples GPUs y grandes cantidades de memoria.

En lugar de entrenar un modelo desde cero, normalmente se utiliza una estrategia llamada *fine tuning*, en la cual un modelo previamente entrenado es adaptado a una tarea específica.

Sin embargo, actualizar todos los parámetros de un LLM sigue siendo extremadamente costoso. Por esta razón se utilizan técnicas más eficientes, como LoRA (*Low-Rank Adaptation*).

La idea general del fine tuning tradicional consiste en modificar completamente la matriz de pesos:

\begin{equation}
W' = W + \Delta W
\end{equation}

donde:
- $W$ representa los pesos originales del modelo,
- $\Delta W$ representa la actualización aprendida durante el entrenamiento.

En LoRA no se modifica directamente toda la matriz $W$. En su lugar, se aprende una aproximación de bajo rango:

\begin{equation}
W' = W + AB
\end{equation}

donde:
- $A$ y $B$ son matrices pequeñas,
- el producto $AB$ representa una corrección de bajo rango,
- y $W$ permanece congelada.

Esto reduce enormemente la memoria requerida,el número de parámetros entrenables y el costo computacional del fine tuning.

En esta práctica utilizaremos:
- PyTorch,
- Transformers de HuggingFace,
- cuantización a 4 bits,
- y LoRA sobre TinyLlama.

In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from datasets import Dataset

from trl import SFTTrainer

# Construcción del dataset de entrenamiento

El modelo será ajustado utilizando un pequeño conjunto de ejemplos relacionados con LoRA.

Cada ejemplo contiene:
- una instrucción del usuario,
- y una respuesta esperada del asistente.

Este formato corresponde al paradigma de *instruction tuning*, ampliamente utilizado en modelos conversacionales modernos.

El objetivo es modificar parcialmente el comportamiento del modelo para que responda adecuadamente preguntas relacionadas con LoRA.

Dado que el dataset es extremadamente pequeño, repetiremos artificialmente los ejemplos múltiples veces. Esto fuerza al modelo a observar repetidamente las mismas relaciones entrada-salida durante el entrenamiento.

En contextos reales se utilizan datasets muchísimo más grandes y diversos. Sin embargo, para fines pedagógicos, un conjunto pequeño permite reducir tiempos de entrenamiento, disminuir el uso de memoria y observar rápidamente el efecto del fine tuning.

In [3]:


data = [
    {
        "text": "<|user|>\nQué es LoRA?\n<|assistant|>\nLoRA es una técnica para ajustar modelos grandes usando matrices de bajo rango."
    },

    {
        "text": "<|user|>\nExplica LoRA.\n<|assistant|>\nLoRA permite adaptar modelos de lenguaje entrenando pocos parámetros adicionales."
    },

    {
        "text": "<|user|>\nPara qué sirve LoRA?\n<|assistant|>\nLoRA reduce el costo computacional del fine tuning de modelos grandes."
    },

    {
        "text": "<|user|>\nQué significa LoRA?\n<|assistant|>\nLoRA significa Low-Rank Adaptation."
    },

    {
        "text": "<|user|>\nCómo funciona LoRA?\n<|assistant|>\nLoRA congela los pesos originales y aprende matrices pequeñas de bajo rango."
    }
]

data = data * 20 #Se están forzando muchas repeticiones de los mismos ejemplos

dataset = Dataset.from_list(data)



# Cuantización y carga del modelo

Los modelos de lenguaje modernos contienen enormes cantidades de parámetros, lo que implica altos requerimientos de memoria. Por ejemplo, si un modelo posee $n$ parámetros y cada parámetro se almacena en precisión de 32 bits, el consumo de memoria aproximado es:

\begin{equation}
\text{Memoria} \approx n \times 4 \text{Bytes}
\end{equation}

Para reducir este costo utilizaremos cuantización a 4 bits, que consiste en representar los pesos usando menos bits, reduciendo así el uso de memoria y permitiendo ejecutar modelos grandes en GPUs más modestas.

Con 4 bits solamente pueden representarse:

\begin{equation}
2^4 = 16
\end{equation}

valores distintos.

Aunque esto reduce precisión numérica, los modelos neuronales modernos poseen una gran redundancia interna y pueden seguir funcionando adecuadamente incluso bajo cuantización agresiva.

También se cargará el tokenizador del modelo, que convierte texto en secuencias numéricas que pueden ser procesadas por el transformer.

Finalmente, se realizará una primera prueba de inferencia antes del fine tuning, con el fin de observar el comportamiento original del modelo.

In [2]:
#Cuantización a 4 bits

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

#Tokenizador

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

#Carga del modelo

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


prompt = """
<|user|>
Qué es LoRA?
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.3,
    do_sample=False
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))




/home/jorge/miniconda3/envs/SI2-LLM/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/jorge/miniconda3/envs/SI2-LLM/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jorge/miniconda3/envs/SI2-LLM/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



<|user|>
Qué es LoRA?
<|assistant|>
LoRA (LoRa) es una tecnología de radio de transmisión de datos de bajo costo desarrollada por la empresa de telecomunicaciones estadounidense Freescale Semiconductor. LoRA es una tecnología de radio de transmisión de datos de bajo costo desarrollada por la empresa de telecomunicaciones estadounidense Freescale Semiconductor. LoRA es


# Configuración de LoRA

Una vez cargado el modelo cuantizado, prepararemos el entrenamiento mediante LoRA.

LoRA introduce pequeñas matrices entrenables dentro de ciertas capas del transformer. En este caso, las modificaciones se realizarán sobre:
- $q\_proj$ (*Query Projection*),
- $v\_proj$ (*Value Projection*).

Estas capas forman parte del mecanismo de atención del transformer.

El parámetro $r$ define el rango de las matrices de adaptación:

\begin{equation}
A \in \mathbb{R}^{d \times r}
\end{equation}

\begin{equation}
B \in \mathbb{R}^{r \times d}
\end{equation}

donde:
- $d$ es la dimensionalidad original,
- y $r$ es un valor pequeño.

El número total de parámetros entrenables disminuye drásticamente, permitiendo realizar fine tuning eficiente incluso en GPUs relativamente pequeñas.

Finalmente, se imprimirá el número total de parámetros, el número de parámetros entrenables y el porcentaje efectivamente ajustado por LoRA.

In [4]:
model = prepare_model_for_kbit_training(model)

#Configuración de LoRA

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "v_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


# Entrenamiento del modelo

Durante el entrenamiento se ajustarán únicamente las matrices introducidas por LoRA, mientras que los pesos originales del modelo permanecerán congelados.

El entrenamiento se realiza mediante gradiente descendente, minimizando una función de pérdida basada en la predicción autoregresiva del siguiente token. 

En modelos de lenguaje autoregresivos, el objetivo consiste en maximizar la probabilidad de la secuencia objetivo:

\begin{equation}
P(x_1, x_2, ..., x_n)
=
\prod_{t=1}^{n}
P(x_t \mid x_1, ..., x_{t-1})
\end{equation}

donde $x_t$ representa el token actual y el modelo predice cada token condicionado a los anteriores.

El parámetro `gradient_accumulation_steps` permite simular batches más grandes acumulando gradientes antes de actualizar los pesos.

In [5]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./tinyllama_lora",
    per_device_train_batch_size=10,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=50,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=256
)

trainer.train()

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/home/jorge/miniconda3/envs/SI2-LLM/lib/python3.11/site-packages/accelerate/accelerator.py:477: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/jorge/miniconda3/envs/SI2-LLM/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,3.290600
2,3.266700
3,3.099300
4,2.977500
5,2.948300
6,2.817700
7,2.725600
8,2.708100
9,2.587900
10,2.464600


TrainOutput(global_step=100, training_loss=0.735668301768601, metrics={'train_runtime': 134.1194, 'train_samples_per_second': 37.28, 'train_steps_per_second': 0.746, 'total_flos': 1093386002595840.0, 'train_loss': 0.735668301768601, 'epoch': 40.0})

# Evaluación del modelo sintonizado

Finalmente, realizaremos nuevamente inferencia sobre el modelo ajustado.

En este caso utilizaremos generación probabilística (`do_sample=True`), permitiendo cierta variabilidad en las respuestas.

La temperatura controla el nivel de aleatoriedad durante la generacióm: temperaturas bajas producen respuestas más determinísticas, temperaturas altas incrementan la diversidad.

El proceso de generación autoregresiva consiste en seleccionar iterativamente el siguiente token:

\begin{equation}
x_t \sim P(x_t \mid x_1, ..., x_{t-1})
\end{equation}

donde cada nuevo token depende de todos los tokens generados anteriormente.

In [7]:
prompt = """
<|user|>
Qué es LoRA?
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# outputs = model.generate(
#     **inputs,
#     max_new_tokens=80,
#     temperature=0.7,
#     do_sample=True
# )

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


<|user|>
Qué es LoRA?
<|assistant|>
LoRA es una técnica para ajustar modelos grandes usando matrices de bajo rango.
</|user|>
¿Cuácula es LoRA?
<|assistant|>
LoRA es una técnica para ajustar modelos grandes usando matrices de bajo rango.
